# Additional main-figure feature plots

This notebook contains additional spatial feature plots used for visualisation of spatial expression of some of the marker genes in Xenium data

## Setup

Imports, plotting defaults, and output paths used throughout the notebook.

In [ ]:
import h5py
import warnings
import os
import spatialdata_plot
from pathlib import Path
import logging
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
from matplotlib.font_manager import FontProperties
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable


In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import anndata as ad
import matplotlib.pyplot as plt
import squidpy as sq
import spatialdata as sd
import spatialdata_io as sio


In [ ]:
import matplotlib as mpl

# Keep text in exported PDFs as real, editable text (not outlined paths) when opened in
# Illustrator, by embedding TrueType fonts instead of converting text to vector shapes.
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"]  = 42


In [ ]:
# Silence noisy font-subsetting debug logs that matplotlib/fontTools emit when saving PDFs.
logging.getLogger("fontTools").setLevel(logging.WARNING)
logging.getLogger("fontTools.subset").setLevel(logging.WARNING)


In [ ]:
# Sanity check: confirm the notebook's working directory, since the relative input/output
# paths used below (out_dir, the input .h5ad file) are resolved relative to it.
!pwd


In [ ]:
# Default directory scanpy writes to when a plotting function is called with save=... .
sc.settings.figdir = "./../../Figures_forpaper/new_colours/noWT3spatial/new_figure"

# Default resolution/background used for figures saved that way.
sc.settings.set_figure_params(dpi_save=300, transparent=True)


In [ ]:
# Directory used below for figures saved manually via plt.savefig()/fig.savefig(); create it
# if missing.
out_dir = Path("./../../Figures_forpaper/new_colours/noWT3spatial/new_figure")
out_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
# Load the annotated, clustered AnnData object (the same combined object used in the
# neighbourhood-analysis notebook).
adata = sc.read_h5ad(
    "../../combined_Xenium_largecells_noWT3_afterclustering_withleiden_annotated_ordered_withTcells_ReannotatedIfgga4.h5ad"
)


## Spatial feature plots

Visualize selected genes directly on tissue coordinates (not via `scanpy`'s built-in spatial
plotting, since these panels need a physical scale bar and, for the multi-sample panels, a
colour scale shared across samples). Four variants follow, each producing its own figure(s):

1. **Per-sample, per-gene plots** for a small marker panel, across every sample individually.
2. A **6-sample grid** for `Nppa` and `F830016B08Rik` ("F8" in the figure), with a shared
   expression scale and one scale bar per figure.
3. The same grid restricted to **3 representative samples**.
4. The same grid generalized to **all 12 samples** in the dataset.

The three grid variants (2–4) share the same approach — compute shared spatial limits and
shared per-gene colour scales across the chosen samples, then lay out one subplot per
sample/gene — and are otherwise independent, so each is self-contained below.


### Per-sample, per-gene spatial plots

For each sample and each gene in `genes`, plot expression on tissue coordinates with a 1 mm
physical scale bar, and save one PDF per sample/gene combination. Each panel is scaled and
coloured independently (no shared axis limits or colour scale across samples/genes — see the
grid panels below for that).


In [ ]:
genes = ["Nppa", "Nppb", "F830016B08Rik", "B2m"]

# Xenium coordinates are in µm, so this is directly 1000 µm = 1 mm
SCALEBAR_UM = 1000

outdir = out_dir

# Define `plot_spatial_gene()` for reuse in the analysis below.
def plot_spatial_gene(
    adata_sample,
    gene,
    sample_name,
    scalebar_um=1000,
    point_size=3
):
    """
    Plot one gene for one Xenium sample with a fixed physical scalebar.
    """

    if gene not in adata_sample.var_names:
        print(f"{gene} not found in {sample_name}")
        return

    # --------------------------------------------------
    # Spatial coordinates
    # Xenium coordinates are already in µm
    # --------------------------------------------------
    coords = adata_sample.obsm["spatial"]

    x = coords[:, 0]
    y = coords[:, 1]

    # --------------------------------------------------
    # Gene expression
    # --------------------------------------------------
    expr = adata_sample[:, gene].X

    if hasattr(expr, "toarray"):
        expr = expr.toarray()

    expr = np.asarray(expr).flatten()

    # --------------------------------------------------
    # Figure
    # --------------------------------------------------
    fig, ax = plt.subplots(figsize=(7, 7))

    # (named `im`, not `sc`, to avoid shadowing the `scanpy as sc` import above)
    im = ax.scatter(
        x,
        y,
        c=expr,
        s=point_size,
        cmap="viridis",
        linewidths=0
    )

    # Correct spatial geometry
    ax.set_aspect("equal")
    ax.invert_yaxis()

    # No grid
    ax.grid(False)

    # --------------------------------------------------
    # Scalebar
    # --------------------------------------------------
    # ax.transData means the bar uses the same units as
    # the spatial coordinates. For Xenium these are µm.
    scalebar_length = scalebar_um

    fontprops = FontProperties(size=10)

    scalebar = AnchoredSizeBar(
        ax.transData,
        scalebar_length,
        f"{scalebar_um} µm",
        loc="lower right",
        pad=0.3,
        color="black",
        frameon=False,
        size_vertical=scalebar_length * 0.01,
        fontproperties=fontprops
    )

    ax.add_artist(scalebar)

    # --------------------------------------------------
    # Labels
    # --------------------------------------------------
    ax.set_title(f"{sample_name} — {gene}")

    ax.set_xlabel("")
    ax.set_ylabel("")

    ax.set_xticks([])
    ax.set_yticks([])

    # Remove frame
    ax.set_frame_on(False)

    for spine in ax.spines.values():
        spine.set_visible(False)

    # Expression colorbar
    cbar = plt.colorbar(
        im,
        ax=ax,
        label=gene,
        fraction=0.046,
        pad=0.04
    )

    plt.tight_layout()

    # --------------------------------------------------
    # Save
    # --------------------------------------------------
    filename = os.path.join(
        outdir,
        f"{sample_name}_{gene}.pdf"
    )

    fig.savefig(
        filename,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(fig)

# ======================================================
# Generate plots for every sample and every gene
# ======================================================

for sample_name in adata.obs["name"].unique():

    adata_sample = adata[
        adata.obs["name"] == sample_name
    ].copy()

    for gene in genes:

        plot_spatial_gene(
            adata_sample,
            gene,
            sample_name,
            scalebar_um=SCALEBAR_UM,
            point_size=3
        )


### Six-sample panel: `Nppa` and `F830016B08Rik` ("F8")

A 2-row × 6-column grid (2 genes × `BE_rep2`, `BE_rep3`, `PBS_rep3`, `PBS_rep2`, `WT_rep4`,
`WT_rep5`), with spatial limits and a per-gene colour scale computed jointly across the six
samples so that panels are directly comparable, plus a single 1 mm scale bar valid for every
panel. Saved as `Nppa_F8_spatial_panel.pdf`.


In [ ]:
# ======================================================
# Settings
# ======================================================

genes = ["Nppa", "F830016B08Rik"]  # "F830016B08Rik" is labelled "F8" in titles/colorbars below

samples = [
    "BE_rep2",
    "BE_rep3",
    "PBS_rep3",
    "PBS_rep2",
    "WT_rep4",
    "WT_rep5",
]

SCALEBAR_UM = 1000  # 1 mm = 1000 µm
POINT_SIZE = 3
CMAP = "viridis"

outdir = out_dir
os.makedirs(outdir, exist_ok=True)

# ======================================================
# Get the six samples
# ======================================================

sample_data = {}

for sample_name in samples:
    if sample_name not in adata.obs["name"].unique():
        print(f"WARNING: {sample_name} not found in adata.obs['name']")
        continue

    sample_data[sample_name] = adata[
        adata.obs["name"] == sample_name
    ].copy()

# ======================================================
# Calculate global spatial limits
#
# This ensures every panel uses exactly the same
# spatial scale, so the 1 mm scalebar represents
# the same physical distance everywhere.
# ======================================================

all_x = []
all_y = []

for sample_name, adata_sample in sample_data.items():

    coords = np.asarray(adata_sample.obsm["spatial"])

    all_x.append(coords[:, 0])
    all_y.append(coords[:, 1])

all_x = np.concatenate(all_x)
all_y = np.concatenate(all_y)

xmin, xmax = np.nanmin(all_x), np.nanmax(all_x)
ymin, ymax = np.nanmin(all_y), np.nanmax(all_y)

# Use the same spatial extent in every panel.
# A small margin is added around the complete dataset.
xpad = (xmax - xmin) * 0.01
ypad = (ymax - ymin) * 0.01

xmin -= xpad
xmax += xpad
ymin -= ypad
ymax += ypad

# ======================================================
# Calculate global expression ranges
#
# One vmin/vmax per gene, calculated across ALL
# six samples. Therefore every sample for a given
# gene uses exactly the same expression scale.
# ======================================================

expression_ranges = {}

for gene in genes:

    gene_values = []

    for sample_name, adata_sample in sample_data.items():

        if gene not in adata_sample.var_names:
            print(f"WARNING: {gene} not found in {sample_name}")
            continue

        expr = adata_sample[:, gene].X

        if hasattr(expr, "toarray"):
            expr = expr.toarray()

        expr = np.asarray(expr).flatten()

        # Keep only finite values
        expr = expr[np.isfinite(expr)]

        if len(expr) > 0:
            gene_values.append(expr)

    if len(gene_values) == 0:
        print(f"WARNING: No expression data found for {gene}")
        continue

    gene_values = np.concatenate(gene_values)

    vmin = np.nanmin(gene_values)
    vmax = np.nanmax(gene_values)

    # Handle the degenerate case where all values are identical
    if vmin == vmax:
        vmax = vmin + 1e-12

    expression_ranges[gene] = (vmin, vmax)

    print(
        f"{gene}: shared expression range = "
        f"{vmin:.4g} - {vmax:.4g}"
    )

# ======================================================
# Create panel
#
# 2 rows:
#   Row 1 = Nppa
#   Row 2 = F8
#
# 6 columns:
#   BE_rep2, BE_rep3, PBS_rep3, PBS_rep2,
#   WT_rep4, WT_rep5
#
# Last column is reserved for the shared colorbars.
# ======================================================

fig = plt.figure(figsize=(24, 8))

gs = fig.add_gridspec(
    nrows=2,
    ncols=7,
    width_ratios=[1, 1, 1, 1, 1, 1, 0.08],
    wspace=0.03,
    hspace=0.08
)

axes = np.empty((2, 6), dtype=object)

for gene_idx, gene in enumerate(genes):

    for sample_idx, sample_name in enumerate(samples):

        ax = fig.add_subplot(
            gs[gene_idx, sample_idx]
        )

        axes[gene_idx, sample_idx] = ax

        # --------------------------------------------------
        # Check sample availability
        # --------------------------------------------------

        if sample_name not in sample_data:
            ax.text(
                0.5,
                0.5,
                f"{sample_name}\nnot found",
                ha="center",
                va="center",
                transform=ax.transAxes
            )
            ax.set_axis_off()
            continue

        adata_sample = sample_data[sample_name]

        # --------------------------------------------------
        # Check gene availability
        # --------------------------------------------------

        if gene not in adata_sample.var_names:
            ax.text(
                0.5,
                0.5,
                f"{gene}\nnot found",
                ha="center",
                va="center",
                transform=ax.transAxes
            )
            ax.set_axis_off()
            continue

        # --------------------------------------------------
        # Spatial coordinates
        # --------------------------------------------------

        coords = np.asarray(
            adata_sample.obsm["spatial"]
        )

        x = coords[:, 0]
        y = coords[:, 1]

        # --------------------------------------------------
        # Gene expression
        # --------------------------------------------------

        expr = adata_sample[:, gene].X

        if hasattr(expr, "toarray"):
            expr = expr.toarray()

        expr = np.asarray(expr).flatten()

        # --------------------------------------------------
        # Shared normalization for this gene
        # --------------------------------------------------

        vmin, vmax = expression_ranges[gene]

        norm = Normalize(
            vmin=vmin,
            vmax=vmax
        )

        # --------------------------------------------------
        # Plot
        # --------------------------------------------------

        ax.scatter(
            x,
            y,
            c=expr,
            s=POINT_SIZE,
            cmap=CMAP,
            norm=norm,
            linewidths=0,
            rasterized=True
        )

        # --------------------------------------------------
        # Identical physical scaling for every panel
        # --------------------------------------------------

        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymax, ymin)  # invert y-axis

        ax.set_aspect("equal")

        # --------------------------------------------------
        # Remove axes
        # --------------------------------------------------

        ax.grid(False)

        ax.set_xticks([])
        ax.set_yticks([])

        ax.set_xlabel("")
        ax.set_ylabel("")

        for spine in ax.spines.values():
            spine.set_visible(False)

        # --------------------------------------------------
        # Sample title
        # --------------------------------------------------

        ax.set_title(
            sample_name,
            fontsize=11,
            pad=5
        )

# ======================================================
# Shared colorbar for Nppa
# ======================================================

if "Nppa" in expression_ranges:

    cax_nppa = fig.add_subplot(gs[0, 6])

    vmin, vmax = expression_ranges["Nppa"]

    norm = Normalize(
        vmin=vmin,
        vmax=vmax
    )

    sm = ScalarMappable(
        norm=norm,
        cmap=CMAP
    )

    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        cax=cax_nppa
    )

    cbar.set_label(
        "Nppa expression",
        fontsize=10
    )

# ======================================================
# Shared colorbar for F8
# ======================================================

if "F830016B08Rik" in expression_ranges:

    cax_f8 = fig.add_subplot(gs[1, 6])

    vmin, vmax = expression_ranges["F830016B08Rik"]

    norm = Normalize(
        vmin=vmin,
        vmax=vmax
    )

    sm = ScalarMappable(
        norm=norm,
        cmap=CMAP
    )

    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        cax=cax_f8
    )

    cbar.set_label(
        "F8 expression",
        fontsize=10
    )

# ======================================================
# ONE single 1 mm scale bar for the entire panel
#
# Because every axis has identical spatial limits and
# identical aspect ratio, this one bar defines the
# physical scale for all six samples in both rows.
# ======================================================

# Put the single scalebar in the bottom-right sample panel
ax_scale = axes[1, 5]

fontprops = FontProperties(size=10)

scalebar = AnchoredSizeBar(
    ax_scale.transData,
    SCALEBAR_UM,
    "1 mm",
    loc="lower right",
    pad=0.4,
    color="black",
    frameon=False,
    size_vertical=SCALEBAR_UM * 0.01,
    fontproperties=fontprops
)

ax_scale.add_artist(scalebar)

# ======================================================
# Row labels
# ======================================================

fig.text(
    0.015,
    0.72,
    "Nppa",
    rotation=90,
    va="center",
    ha="center",
    fontsize=13,
    fontweight="bold"
)

fig.text(
    0.015,
    0.28,
    "F8",
    rotation=90,
    va="center",
    ha="center",
    fontsize=13,
    fontweight="bold"
)

# ======================================================
# Save
# ======================================================

plt.subplots_adjust(
    left=0.035,
    right=0.98,
    top=0.92,
    bottom=0.05
)

filename = os.path.join(
    outdir,
    "Nppa_F8_spatial_panel.pdf"
)

# Save the completed figure to disk.
fig.savefig(
    filename,
    dpi=300,
    bbox_inches="tight"
)

# Display the completed figure.
plt.show()
plt.close(fig)

print(f"Saved panel to: {filename}")


### Three-sample panel (subset)

The same grid layout as above, restricted to three representative samples
(`BE_rep1`, `PBS_rep2`, `WT_rep4` — one per condition), for a more compact figure. Saved as
`Nppa_F8_spatial_panel_lesssamples.pdf`.


In [ ]:
# ======================================================
# Settings
# ======================================================

genes = ["Nppa", "F830016B08Rik"]  # "F830016B08Rik" is labelled "F8" in titles/colorbars below

samples = [
    "BE_rep1",
    "PBS_rep2",
    "WT_rep4"
]

SCALEBAR_UM = 1000  # 1 mm = 1000 µm
POINT_SIZE = 3
CMAP = "viridis"

outdir = out_dir
os.makedirs(outdir, exist_ok=True)

# ======================================================
# Get the six samples
# ======================================================

sample_data = {}

for sample_name in samples:
    if sample_name not in adata.obs["name"].unique():
        print(f"WARNING: {sample_name} not found in adata.obs['name']")
        continue

    sample_data[sample_name] = adata[
        adata.obs["name"] == sample_name
    ].copy()

# ======================================================
# Calculate global spatial limits
#
# This ensures every panel uses exactly the same
# spatial scale, so the 1 mm scalebar represents
# the same physical distance everywhere.
# ======================================================

all_x = []
all_y = []

for sample_name, adata_sample in sample_data.items():

    coords = np.asarray(adata_sample.obsm["spatial"])

    all_x.append(coords[:, 0])
    all_y.append(coords[:, 1])

all_x = np.concatenate(all_x)
all_y = np.concatenate(all_y)

xmin, xmax = np.nanmin(all_x), np.nanmax(all_x)
ymin, ymax = np.nanmin(all_y), np.nanmax(all_y)

# Use the same spatial extent in every panel.
# A small margin is added around the complete dataset.
xpad = (xmax - xmin) * 0.01
ypad = (ymax - ymin) * 0.01

xmin -= xpad
xmax += xpad
ymin -= ypad
ymax += ypad

# ======================================================
# Calculate global expression ranges
#
# One vmin/vmax per gene, calculated across ALL
# six samples. Therefore every sample for a given
# gene uses exactly the same expression scale.
# ======================================================

expression_ranges = {}

for gene in genes:

    gene_values = []

    for sample_name, adata_sample in sample_data.items():

        if gene not in adata_sample.var_names:
            print(f"WARNING: {gene} not found in {sample_name}")
            continue

        expr = adata_sample[:, gene].X

        if hasattr(expr, "toarray"):
            expr = expr.toarray()

        expr = np.asarray(expr).flatten()

        # Keep only finite values
        expr = expr[np.isfinite(expr)]

        if len(expr) > 0:
            gene_values.append(expr)

    if len(gene_values) == 0:
        print(f"WARNING: No expression data found for {gene}")
        continue

    gene_values = np.concatenate(gene_values)

    vmin = np.nanmin(gene_values)
    vmax = np.nanmax(gene_values)

    # Handle the degenerate case where all values are identical
    if vmin == vmax:
        vmax = vmin + 1e-12

    expression_ranges[gene] = (vmin, vmax)

    print(
        f"{gene}: shared expression range = "
        f"{vmin:.4g} - {vmax:.4g}"
    )

# ======================================================
# Create panel
#
# 2 rows:
#   Row 1 = Nppa
#   Row 2 = F830016B08Rik
#
# 4 columns:
#   BE_rep1, PBS_rep2, WT_rep4
#   + last column reserved for shared colorbars
# ======================================================

fig = plt.figure(figsize=(13, 8))

gs = fig.add_gridspec(
    nrows=2,
    ncols=4,
    width_ratios=[1, 1, 1, 0.08],
    wspace=0.03,
    hspace=0.08
)

axes = np.empty((2, 3), dtype=object)

for gene_idx, gene in enumerate(genes):

    for sample_idx, sample_name in enumerate(samples):

        ax = fig.add_subplot(
            gs[gene_idx, sample_idx]
        )

        axes[gene_idx, sample_idx] = ax

        # --------------------------------------------------
        # Check sample availability
        # --------------------------------------------------

        if sample_name not in sample_data:
            ax.text(
                0.5,
                0.5,
                f"{sample_name}\nnot found",
                ha="center",
                va="center",
                transform=ax.transAxes
            )
            ax.set_axis_off()
            continue

        adata_sample = sample_data[sample_name]

        # --------------------------------------------------
        # Check gene availability
        # --------------------------------------------------

        if gene not in adata_sample.var_names:
            ax.text(
                0.5,
                0.5,
                f"{gene}\nnot found",
                ha="center",
                va="center",
                transform=ax.transAxes
            )
            ax.set_axis_off()
            continue

        # --------------------------------------------------
        # Spatial coordinates
        # --------------------------------------------------

        coords = np.asarray(adata_sample.obsm["spatial"])

        x = coords[:, 0]
        y = coords[:, 1]

        # --------------------------------------------------
        # Gene expression
        # --------------------------------------------------

        expr = adata_sample[:, gene].X

        if hasattr(expr, "toarray"):
            expr = expr.toarray()

        expr = np.asarray(expr).flatten()

        # --------------------------------------------------
        # Shared normalization for this gene
        # --------------------------------------------------

        vmin, vmax = expression_ranges[gene]

        norm = Normalize(
            vmin=vmin,
            vmax=vmax
        )

        # --------------------------------------------------
        # Plot
        # --------------------------------------------------

        ax.scatter(
            x,
            y,
            c=expr,
            s=POINT_SIZE,
            cmap=CMAP,
            norm=norm,
            linewidths=0,
            rasterized=True
        )

        # --------------------------------------------------
        # Identical physical scaling
        # --------------------------------------------------

        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymax, ymin)
        ax.set_aspect("equal")

        # --------------------------------------------------
        # Remove axes
        # --------------------------------------------------

        ax.grid(False)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xlabel("")
        ax.set_ylabel("")

        for spine in ax.spines.values():
            spine.set_visible(False)

        # Only show sample names on the top row
        if gene_idx == 0:
            ax.set_title(
                sample_name,
                fontsize=11,
                pad=5
            )
    
# ======================================================
# Shared colorbar for Nppa
# ======================================================

if "Nppa" in expression_ranges:

    cax_nppa = fig.add_subplot(gs[0, 3])

    vmin, vmax = expression_ranges["Nppa"]

    sm = ScalarMappable(
        norm=Normalize(vmin=vmin, vmax=vmax),
        cmap=CMAP
    )
    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        cax=cax_nppa
    )

    cbar.set_label(
        "Nppa expression",
        fontsize=10
    )

# ======================================================
# Shared colorbar for F830016B08Rik
# ======================================================

if "F830016B08Rik" in expression_ranges:

    cax_f8 = fig.add_subplot(gs[1, 3])

    vmin, vmax = expression_ranges["F830016B08Rik"]

    sm = ScalarMappable(
        norm=Normalize(vmin=vmin, vmax=vmax),
        cmap=CMAP
    )
    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        cax=cax_f8
    )

    cbar.set_label(
        "F8 expression",
        fontsize=10
    )

# ======================================================
# ONE shared 1 mm scale bar
#
# Put it in the bottom-right sample:
# F8 / WT_rep4
# ======================================================

ax_scale = axes[1, 2]

fontprops = FontProperties(size=10)

scalebar = AnchoredSizeBar(
    ax_scale.transData,
    SCALEBAR_UM,          # 1000 µm = 1 mm
    "1 mm",
    loc="lower right",
    pad=0.4,
    borderpad=0.5,
    sep=4,
    color="black",
    frameon=False,
    size_vertical=SCALEBAR_UM * 0.01,
    fontproperties=fontprops
)

ax_scale.add_artist(scalebar)

# ======================================================
# Gene names on left side
# ======================================================

axes[0, 0].text(
    -0.08,
    0.5,
    "Nppa",
    transform=axes[0, 0].transAxes,
    rotation=90,
    va="center",
    ha="center",
    fontsize=13,
    fontweight="bold",
    clip_on=False
)

axes[1, 0].text(
    -0.08,
    0.5,
    "F8",
    transform=axes[1, 0].transAxes,
    rotation=90,
    va="center",
    ha="center",
    fontsize=13,
    fontweight="bold",
    clip_on=False
)

# ======================================================
# Final layout
# ======================================================

plt.subplots_adjust(
    left=0.08,
    right=0.95,
    top=0.92,
    bottom=0.06
)

# ======================================================
# Save
# ======================================================

filename = os.path.join(
    outdir,
    "Nppa_F8_spatial_panel_lesssamples.pdf"
)

# Save the completed figure to disk.
fig.savefig(
    filename,
    dpi=300,
    bbox_inches="tight"
)

# Display the completed figure.
plt.show()
plt.close(fig)

print(f"Saved panel to: {filename}")


### All-samples panel

The same grid layout, generalized to every sample in `adata` (12 samples after `WT_rep3`
exclusion), picked up automatically via `adata.obs["name"].unique()`. An explicit list of the
12 sample names is kept commented out above as a documented, easy-to-restore alternative if
`adata` ever contains extra samples you don't want included. Saved as
`Nppa_F8_spatial_panel_all_12_samples.pdf`.


In [ ]:
# ======================================================
# Settings
# ======================================================

genes = ["Nppa", "F830016B08Rik"]  # "F830016B08Rik" is labelled "F8" in titles/colorbars below

# ------------------------------------------------------
# OPTION 1: explicitly list your 12 samples
# ------------------------------------------------------
# samples = [
#     "BE_rep1",
#     "BE_rep2",
#     "BE_rep3",
#     "BE_rep4",
#     "PBS_rep1",
#     "PBS_rep2",
#     "PBS_rep3",
#     "PBS_rep4",
#     "WT_rep1",
#     "WT_rep2",
#     "WT_rep4",
#     "WT_rep5",
# ]

# ------------------------------------------------------
# OPTION 2: automatically use ALL samples in adata
# ------------------------------------------------------
samples = list(adata.obs["name"].unique())

print(f"Number of samples: {len(samples)}")
print(samples)

SCALEBAR_UM = 1000  # 1 mm = 1000 µm
POINT_SIZE = 3
CMAP = "viridis"

outdir = out_dir
os.makedirs(outdir, exist_ok=True)

n_samples = len(samples)
n_genes = len(genes)

# ======================================================
# Get all samples
# ======================================================

sample_data = {}

for sample_name in samples:

    if sample_name not in adata.obs["name"].unique():
        print(f"WARNING: {sample_name} not found in adata.obs['name']")
        continue

    sample_data[sample_name] = adata[
        adata.obs["name"] == sample_name
    ].copy()

# ======================================================
# Calculate global spatial limits
#
# All 12 samples use exactly the same x/y limits.
# Therefore spatial scale is identical in every panel.
# ======================================================

all_x = []
all_y = []

for sample_name, adata_sample in sample_data.items():

    coords = np.asarray(adata_sample.obsm["spatial"])

    all_x.append(coords[:, 0])
    all_y.append(coords[:, 1])

all_x = np.concatenate(all_x)
all_y = np.concatenate(all_y)

xmin, xmax = np.nanmin(all_x), np.nanmax(all_x)
ymin, ymax = np.nanmin(all_y), np.nanmax(all_y)

# Small margin
xpad = (xmax - xmin) * 0.01
ypad = (ymax - ymin) * 0.01

xmin -= xpad
xmax += xpad
ymin -= ypad
ymax += ypad

# ======================================================
# Calculate global expression ranges
#
# One shared vmin/vmax per gene across ALL 12 samples.
# ======================================================

expression_ranges = {}

for gene in genes:

    gene_values = []

    for sample_name, adata_sample in sample_data.items():

        if gene not in adata_sample.var_names:
            print(f"WARNING: {gene} not found in {sample_name}")
            continue

        expr = adata_sample[:, gene].X

        if hasattr(expr, "toarray"):
            expr = expr.toarray()

        expr = np.asarray(expr).flatten()

        # Keep finite values only
        expr = expr[np.isfinite(expr)]

        if len(expr) > 0:
            gene_values.append(expr)

    if len(gene_values) == 0:
        print(f"WARNING: No expression data found for {gene}")
        continue

    gene_values = np.concatenate(gene_values)

    vmin = np.nanmin(gene_values)
    vmax = np.nanmax(gene_values)

    # Avoid zero-width normalization
    if vmin == vmax:
        vmax = vmin + 1e-12

    expression_ranges[gene] = (vmin, vmax)

    print(
        f"{gene}: shared expression range = "
        f"{vmin:.4g} - {vmax:.4g}"
    )

# ======================================================
# Create panel
#
# 2 rows:
#   Row 1 = Nppa
#   Row 2 = F8
#
# 12 columns:
#   one column per sample
#
# Final narrow column:
#   shared colorbars
# ======================================================

# Scale figure width automatically with number of samples
fig_width = 3.2 * n_samples + 1.2
fig_height = 8

fig = plt.figure(
    figsize=(fig_width, fig_height)
)

gs = fig.add_gridspec(
    nrows=n_genes,
    ncols=n_samples + 1,
    width_ratios=[1] * n_samples + [0.08],
    wspace=0.03,
    hspace=0.08
)

axes = np.empty(
    (n_genes, n_samples),
    dtype=object
)

# ======================================================
# Plot samples
# ======================================================

for gene_idx, gene in enumerate(genes):

    for sample_idx, sample_name in enumerate(samples):

        ax = fig.add_subplot(
            gs[gene_idx, sample_idx]
        )

        axes[gene_idx, sample_idx] = ax

        # --------------------------------------------------
        # Check sample availability
        # --------------------------------------------------

        if sample_name not in sample_data:

            ax.text(
                0.5,
                0.5,
                f"{sample_name}\nnot found",
                ha="center",
                va="center",
                transform=ax.transAxes
            )

            ax.set_axis_off()
            continue

        adata_sample = sample_data[sample_name]

        # --------------------------------------------------
        # Check gene availability
        # --------------------------------------------------

        if gene not in adata_sample.var_names:

            ax.text(
                0.5,
                0.5,
                f"{gene}\nnot found",
                ha="center",
                va="center",
                transform=ax.transAxes
            )

            ax.set_axis_off()
            continue

        # --------------------------------------------------
        # Spatial coordinates
        # --------------------------------------------------

        coords = np.asarray(
            adata_sample.obsm["spatial"]
        )

        x = coords[:, 0]
        y = coords[:, 1]

        # --------------------------------------------------
        # Gene expression
        # --------------------------------------------------

        expr = adata_sample[:, gene].X

        if hasattr(expr, "toarray"):
            expr = expr.toarray()

        expr = np.asarray(expr).flatten()

        # --------------------------------------------------
        # Shared normalization for this gene
        # --------------------------------------------------

        vmin, vmax = expression_ranges[gene]

        norm = Normalize(
            vmin=vmin,
            vmax=vmax
        )

        # --------------------------------------------------
        # Plot
        # --------------------------------------------------

        ax.scatter(
            x,
            y,
            c=expr,
            s=POINT_SIZE,
            cmap=CMAP,
            norm=norm,
            linewidths=0,
            rasterized=True
        )

        # --------------------------------------------------
        # Identical physical scaling for EVERY sample
        # --------------------------------------------------

        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymax, ymin)

        ax.set_aspect("equal")

        # --------------------------------------------------
        # Remove axes
        # --------------------------------------------------

        ax.grid(False)

        ax.set_xticks([])
        ax.set_yticks([])

        ax.set_xlabel("")
        ax.set_ylabel("")

        for spine in ax.spines.values():
            spine.set_visible(False)

        # --------------------------------------------------
        # Sample titles
        #
        # Only put them on the top row to avoid repeating
        # the same sample name twice.
        # --------------------------------------------------

        if gene_idx == 0:

            ax.set_title(
                sample_name,
                fontsize=10,
                pad=5,
                rotation=45,
                ha="left"
            )

# ======================================================
# Shared colorbar for Nppa
# ======================================================

if "Nppa" in expression_ranges:

    cax_nppa = fig.add_subplot(
        gs[0, n_samples]
    )

    vmin, vmax = expression_ranges["Nppa"]

    norm = Normalize(
        vmin=vmin,
        vmax=vmax
    )

    sm = ScalarMappable(
        norm=norm,
        cmap=CMAP
    )

    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        cax=cax_nppa
    )

    cbar.set_label(
        "Nppa expression",
        fontsize=10
    )

# ======================================================
# Shared colorbar for F8
# ======================================================

if "F830016B08Rik" in expression_ranges:

    cax_f8 = fig.add_subplot(
        gs[1, n_samples]
    )

    vmin, vmax = expression_ranges["F830016B08Rik"]

    norm = Normalize(
        vmin=vmin,
        vmax=vmax
    )

    sm = ScalarMappable(
        norm=norm,
        cmap=CMAP
    )

    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        cax=cax_f8
    )

    cbar.set_label(
        "F8 expression",
        fontsize=10
    )

# ======================================================
# ONE single 1 mm scale bar
#
# Because every panel has identical x/y limits and
# aspect ratio, the scale bar applies to ALL panels.
#
# Put it in the final sample of the bottom row.
# ======================================================

ax_scale = axes[
    n_genes - 1,
    n_samples - 1
]

fontprops = FontProperties(
    size=10
)

scalebar = AnchoredSizeBar(
    ax_scale.transData,
    SCALEBAR_UM,
    "1 mm",
    loc="lower right",
    pad=0.4,
    color="black",
    frameon=False,
    size_vertical=SCALEBAR_UM * 0.01,
    fontproperties=fontprops
)

ax_scale.add_artist(scalebar)

# ======================================================
# Row labels
# ======================================================

fig.text(
    0.012,
    0.72,
    "Nppa",
    rotation=90,
    va="center",
    ha="center",
    fontsize=13,
    fontweight="bold"
)

fig.text(
    0.012,
    0.28,
    "F8",
    rotation=90,
    va="center",
    ha="center",
    fontsize=13,
    fontweight="bold"
)

# ======================================================
# Save
# ======================================================

plt.subplots_adjust(
    left=0.025,
    right=0.985,
    top=0.88,
    bottom=0.04
)

filename = os.path.join(
    outdir,
    "Nppa_F8_spatial_panel_all_12_samples.pdf"
)

# Save the completed figure to disk.
fig.savefig(
    filename,
    dpi=300,
    bbox_inches="tight"
)

# Display the completed figure.
plt.show()
plt.close(fig)

print(f"Saved panel to: {filename}")
